In [4]:
import sys

!{sys.executable} -m ensurepip --upgrade
!{sys.executable} -m pip install .

Looking in links: c:\Users\shrir\AppData\Local\Temp\tmprkkzf5bp
Processing .\.
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for house-pricing: filename=house_pricing-0.1.0-py3-none-any.whl size=12496 sha256=e33b608ba336a3e57f2b8c272a48e12bcf5696d19998a35ca445bdc54c32e1d8
  Stored in directory: C:\Users\shrir\AppData\Local\Temp\pip-ephem-wheel-cache-43xc4lvy\wheels\b5\6e\fe\713577e68cdd81b2c9361e66506cdffe540627098148da5723
Successfully built house-pricing
  Attempting uninstall: house-pricing
    Found existing installation: house-pricing 0.1.0
    Uninstalling house-pricing-0.1.0:
      Successfully uninstalled house-pricing-0.1.0


In [2]:
import pandas as pd
from house_cleaning import remove_bad_data, build_cleaning_pipeline

In [2]:
from pathlib import Path

filename = "King_County_House_prices_dataset.csv"
data_path = next(
    (
        parent / "data" / filename
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "data" / filename).is_file()
    ),
    None,
)
if data_path is None:
    raise FileNotFoundError(f"Could not find {filename!r} in a data directory under {Path.cwd()} or its parents.")

df = pd.read_csv(data_path)

In [17]:
df_valid = remove_bad_data(df,15)

In [18]:
df_valid.shape


(21596, 21)

In [16]:
df.shape

(21597, 21)

In [19]:
df_clean = build_cleaning_pipeline().fit_transform(df_valid)

In [20]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 21596 entries, 0 to 21596
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id                 21596 non-null  int64         
 1   date               21596 non-null  datetime64[us]
 2   price              21596 non-null  float64       
 3   bedrooms           21596 non-null  int64         
 4   bathrooms          21596 non-null  float64       
 5   sqft_living        21596 non-null  int64         
 6   sqft_lot           21596 non-null  int64         
 7   floors             21596 non-null  float64       
 8   waterfront         19220 non-null  float64       
 9   view               21533 non-null  float64       
 10  condition          21596 non-null  int64         
 11  grade              21596 non-null  int64         
 12  sqft_above         21596 non-null  int64         
 13  sqft_basement      21596 non-null  int64         
 14  zipcode            215

In [21]:
df_clean.isna().sum()

id                      0
date                    0
price                   0
bedrooms                0
bathrooms               0
sqft_living             0
sqft_lot                0
floors                  0
waterfront           2376
view                   63
condition               0
grade                   0
sqft_above              0
sqft_basement           0
zipcode                 0
lat                     0
long                    0
sqft_living15           0
sqft_lot15              0
last_known_change       0
dtype: int64

In [1]:
%load_ext autoreload 
%autoreload 2

In [5]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNetCV, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold, cross_validate, train_test_split

from house_pricing import remove_bad_data, build_model_pipeline


from pathlib import Path

filename = "King_County_House_prices_dataset.csv"
data_path = next(
    (
        parent / "data" / filename
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "data" / filename).is_file()
    ),
    None,
)
if data_path is None:
    raise FileNotFoundError(f"Could not find {filename!r} in a data folder under the current directory or its parents.")

df = remove_bad_data(pd.read_csv(data_path),15)

X = df.drop(columns="price")
y = df["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Compare models with cross-validation on TRAIN only ---
candidates = {
  "ridge":               Ridge(alpha=1.0),
          # Tunes alpha and l1_ratio with its own 5-fold CV, inside each training fold
   "elastic_net":         ElasticNetCV(l1_ratio=[0.2, 0.5, 0.8, 1.0],
                                              alphas=[0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03],
                                              cv=5, max_iter=5000, n_jobs=-1),
   "random_forest":       RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=42),
   "hist_gradient_boost": HistGradientBoostingRegressor(random_state=42),
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)

results = []
for name, reg in candidates.items():
    scores = cross_validate(build_model_pipeline(reg), X_train, y_train, cv=cv,
                            scoring={"rmse": "neg_root_mean_squared_error",
                                     "mae":  "neg_mean_absolute_error",
                                     "r2":   "r2"})
    results.append({"model": name,
                    "cv_rmse": -scores["test_rmse"].mean(),
                    "cv_mae":  -scores["test_mae"].mean(),
                    "cv_r2":   scores["test_r2"].mean()})

results = pd.DataFrame(results).sort_values("cv_rmse")
print(results)

# --- Fit the best model on all training data, evaluate on TEST once ---
best_name = results.iloc[0]["model"]
final_model = build_model_pipeline(candidates[best_name]).fit(X_train, y_train)

pred = final_model.predict(X_test)
print(f"{best_name}  RMSE: {root_mean_squared_error(y_test, pred):,.0f}  "
      f"MAE: {mean_absolute_error(y_test, pred):,.0f}  R²: {r2_score(y_test, pred):.3f}")

                 model        cv_rmse        cv_mae     cv_r2
3  hist_gradient_boost  128733.371074  68611.696052  0.877200
2        random_forest  138555.590349  73599.979738  0.857791
1          elastic_net  166083.688782  76840.226280  0.787441
0                ridge  166231.608576  76826.457757  0.786876
hist_gradient_boost  RMSE: 129,674  MAE: 67,760  R²: 0.872


In [3]:
from scipy.stats import loguniform, randint
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, RandomizedSearchCV

cv = KFold(n_splits=5, shuffle=True, random_state=42)
p = "model__regressor__"

param_dist = {
    f"{p}learning_rate":     loguniform(0.02, 0.2),
    f"{p}max_iter":          randint(200, 1000),
    f"{p}max_leaf_nodes":    randint(15, 64),
    f"{p}min_samples_leaf":  randint(10, 80),
    f"{p}l2_regularization": loguniform(1e-3, 10),    
}

search = RandomizedSearchCV(
    build_model_pipeline(HistGradientBoostingRegressor(random_state=42)),
    param_dist, n_iter=40, cv=cv,
    scoring="neg_root_mean_squared_error", n_jobs=-1, random_state=42,
)
search.fit(X_train, y_train)
print(search.best_params_, -search.best_score_)

{'model__regressor__l2_regularization': np.float64(0.015314189711654769), 'model__regressor__learning_rate': np.float64(0.029261523555583403), 'model__regressor__max_iter': 720, 'model__regressor__max_leaf_nodes': 38, 'model__regressor__min_samples_leaf': 10} 125787.16173607433
